In [38]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
import IPython.display as ipd

In [5]:
import math
from numpy.linalg import norm, svd

def RPCA(X, lmbda=.01, tol=1e-7, maxiter=1000, verbose=True):
  Y = X
  norm_two = norm(Y.ravel(), 2)
  norm_inf = norm(Y.ravel(), np.inf) / lmbda
  dual_norm = np.max([norm_two, norm_inf])

  Y = Y / dual_norm
  A = np.zeros(Y.shape)
  E = np.zeros(Y.shape)
  dnorm = norm(X, 'fro')

  mu = 1.25 / norm_two
  rho = 1.5
  sv = 10.
  n = Y.shape[0]
  itr = 0

  while True:
    Eraw = X - A + (1 / mu) * Y
    Eupdate = np.maximum(Eraw - lmbda / mu, 0) + np.minimum(Eraw + lmbda / mu, 0)

    U, S, V = svd(X - Eupdate + (1 / mu) * Y, full_matrices=False)

    svp = (S > 1 / mu).shape[0]
    if svp < sv:
      sv = np.min([svp + 1, n])
    else:
      sv = np.min([svp + round(.05 * n), n])
    Aupdate = np.dot(np.dot(U[:, :svp], np.diag(S[:svp] - 1 / mu)), V[:svp, :])

    A = Aupdate
    E = Eupdate
    Z = X - A - E
    Y = Y + mu * Z

    mu = np.min([mu * rho, mu * 1e7])
    itr += 1
    if ((norm(Z, 'fro') / dnorm) < tol) or (itr >= maxiter):
      break

  if verbose:
    print("Finished at iteration %d" % (itr))  

  return A, E

def singing_voice_separation(X, lmbda=1, nFFT=2048, hopLength=512, gain=1.5, power=1):
  scf = 2 / 3.0
  S_mix = scf * librosa.stft(X, n_fft=nFFT, hop_length=hopLength)  # short-time Fourier transform

  A_mag, E_mag = RPCA(np.power(np.abs(S_mix), power), lmbda=lmbda / math.sqrt(max(S_mix.shape)))
  PHASE = np.angle(S_mix)

  A = A_mag * np.exp(1j * PHASE)
  E = E_mag * np.exp(1j * PHASE)

  mask = np.abs(E) > (gain * np.abs(A))
  Emask = mask * S_mix
  Amask = S_mix - Emask

  wavoutE = librosa.istft(Emask, hop_length=hopLength)
  wavoutA = librosa.istft(Amask, hop_length=hopLength)

  wavoutE /= np.abs(wavoutE).max()
  wavoutA /= np.abs(wavoutA).max()

  return np.array([wavoutE, wavoutA])

Exer2 - music1: (10s - 14s)

nfft = 2048, hopLength = 256, lmbda = 1, SDR = 55

---
Exer2 - music2: (32.5s - 36.5s)

nfft = 2048, hopLength = 256, lmbda = 0.9, SDR = 53

nfft = 4096, hopLength = 1024, lmbda = 1.1, SDR = 52

---
Exer2 - music3:

nfft = 2048, hopLength = 256, lmbda = 1.2, SDR = 38 (20s - 24s)

nfft = 2048, hopLength = 256, lmbda = 1.25, SDR = 44 (20s - 24s)

In [327]:
# Load the sample music
file2 = "./Exercises Dataset/Exer2 - music2.mp3"

sample2, sr2 = librosa.load(
  file2,
  mono=True,
  sr=None,
  # offset=32.5,
  # duration=4.0
)

ipd.Audio(data=sample2, rate=sr2)

In [323]:

import soundfile as sf

sf.write("Exer2 - music2.mp3", sample2.T, sr2)

In [337]:
import mir_eval
import warnings

# Apply RPCA
lmbda = 1.1
nFFT = 4096
# hopLength = int(nFFT / 4)
hopLength = 1024

separated_sources_RPCA = singing_voice_separation(
  sample2,
  lmbda=lmbda,
  nFFT=nFFT,
  hopLength=hopLength
)

warnings.simplefilter("ignore")

reference = np.array(sample2) # shape (n_samples)
estimated_RPCA = np.mean(separated_sources_RPCA, axis=0)

min_len = min([len(reference), len(estimated_RPCA)])

reference = reference[:min_len]
estimated_RPCA = estimated_RPCA[:min_len]

sdr, sir, sar, _ = mir_eval.separation.bss_eval_sources(
  reference,
  estimated_RPCA
)

print("RPCA")
print("SDR:", sdr)
print("SIR:", sir)
print("SAR:", sar)

Finished at iteration 47
RPCA
SDR: [52.80530556]
SIR: [inf]
SAR: [52.80530556]


In [298]:
ipd.display(ipd.Audio(data=separated_sources_RPCA[0], rate=sr2))
ipd.display(ipd.Audio(data=separated_sources_RPCA[1], rate=sr2))